In [49]:
!pip install ultralytics --quiet
!pip install torchvision --quiet
!pip install torch --quiet
!pip install pillow --quiet
!pip install scikit-learn --quiet

In [50]:
from PIL import Image
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Subset, DataLoader
from pathlib import Path

import torchvision.transforms.v2 as transforms_v2
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [51]:
BASE_DIR = Path.cwd().parents[0]
CSV_FN = 'manifest.csv'
CSV_PATH = BASE_DIR / 'CVModel-Data_Preparation' / 'Labeling_Preprocessing' / CSV_FN

In [52]:
class CSVDataset(Dataset):

    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform

        # label: string to number
        self.label_map = {label: idx for idx, label in enumerate(self.data['label'].unique())}
        self.data['label'] = self.data['label'].map(self.label_map)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image']
        label = self.data.iloc[idx]['label']

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label
    

In [53]:
train_transform = transforms_v2.Compose([
    transforms_v2.RandomResizedCrop(224),
    transforms_v2.RandomHorizontalFlip(),
    transforms_v2.ColorJitter(0.15),
    transforms_v2.ToTensor(),
])

val_transform = transforms_v2.Compose([
    transforms_v2.Resize(256),
    transforms_v2.CenterCrop(224),
    transforms_v2.ToTensor(),
])

/opt/anaconda3/envs/sscvmodel-fine_tuning/lib/python3.12/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [54]:
full_dataset_train = CSVDataset(CSV_PATH, transform=train_transform)
full_dataset_val = CSVDataset(CSV_PATH, transform=val_transform)

In [55]:
indices = np.arange(len(full_dataset_train)) # they are same size
labels = full_dataset_train.data['label']

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [56]:
train_dataset = Subset(full_dataset_train, train_indices)
val_dataset = Subset(full_dataset_val, val_indices)

In [57]:
len(train_dataset), len(val_dataset)

(146, 37)

In [58]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

In [59]:
next(iter(train_loader))

[tensor([[[[1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9843],
           [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9843],
           [1.0000, 1.0000, 0.9922,  ..., 0.9843, 0.9843, 0.9843],
           ...,
           [0.5255, 0.5804, 0.6196,  ..., 0.3922, 0.3961, 0.3961],
           [0.4824, 0.4863, 0.5059,  ..., 0.4000, 0.3961, 0.3765],
           [0.4157, 0.3843, 0.3765,  ..., 0.4000, 0.3922, 0.3647]],
 
          [[1.0000, 1.0000, 0.9922,  ..., 0.9922, 0.9922, 0.9882],
           [1.0000, 1.0000, 0.9922,  ..., 0.9922, 0.9922, 0.9882],
           [0.9922, 0.9922, 0.9882,  ..., 0.9882, 0.9882, 0.9882],
           ...,
           [0.5843, 0.6196, 0.6314,  ..., 0.4824, 0.4863, 0.4824],
           [0.5216, 0.5098, 0.5020,  ..., 0.4902, 0.4863, 0.4706],
           [0.4471, 0.3961, 0.3608,  ..., 0.4902, 0.4824, 0.4549]],
 
          [[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.00

In [60]:
yolo = YOLO('yolov8n-cls.pt')  
model = yolo.model 

In [61]:
num_classes = len(full_dataset_train.data['label'].unique())
model.model[-1].linear = nn.Linear(model.model[-1].linear.in_features, num_classes)

In [62]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
model.to(device)

ClassificationModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
   

In [63]:
epochs = 100

In [64]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

In [65]:
WEIGHTS_DIR = Path.cwd().parent / 'src'
WEIGHTS_PATH = WEIGHTS_DIR / 'best_model.pt'

In [67]:
idx_to_label = {idx: label for label, idx in full_dataset_train.label_map.items()}
print(idx_to_label)

best_val_acc = 0
patience = 15
patience_counter = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # ---- VALIDATE ----
    model.eval()
    correct = 0
    total = 0
    misclassified_pairs = {}

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            for true_idx, pred_idx in zip(labels.cpu().tolist(), preds.cpu().tolist()):
                if true_idx != pred_idx:
                    key = (true_idx, pred_idx)
                    misclassified_pairs[key] = misclassified_pairs.get(key, 0) + 1

    val_acc = correct / total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), WEIGHTS_PATH)
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print('Early stopping triggered')
        break

    scheduler.step()

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")

    if epoch == epochs - 1:
        if misclassified_pairs:
            print('Misclassified classes (true -> predicted) on final epoch:')
            for (true_idx, pred_idx), count in sorted(misclassified_pairs.items(), key=lambda x: x[1], reverse=True):
                true_label = idx_to_label.get(true_idx, str(true_idx))
                pred_label = idx_to_label.get(pred_idx, str(pred_idx))
                print(f"  {true_label} -> {pred_label}: {count}")
        else:
            print('No misclassifications in validation on final epoch.')


{0: 'football', 1: 'baseball', 2: 'soccer', 3: 'pingpong', 4: 'basketball', 5: 'petanque', 6: 'volleyball', 7: 'tennis'}
Epoch 1
Train Loss: 0.9322
Val Accuracy: 0.6216
Epoch 2
Train Loss: 0.8515
Val Accuracy: 0.6757
Epoch 3
Train Loss: 0.8992
Val Accuracy: 0.6216
Epoch 4
Train Loss: 0.8146
Val Accuracy: 0.5405
Epoch 5
Train Loss: 0.8912
Val Accuracy: 0.6486
Epoch 6
Train Loss: 0.8360
Val Accuracy: 0.6216
Epoch 7
Train Loss: 0.8532
Val Accuracy: 0.5676
Epoch 8
Train Loss: 1.1028
Val Accuracy: 0.5946
Epoch 9
Train Loss: 0.7335
Val Accuracy: 0.6486
Epoch 10
Train Loss: 0.8422
Val Accuracy: 0.6757
Epoch 11
Train Loss: 0.7829
Val Accuracy: 0.6216
Epoch 12
Train Loss: 0.8335
Val Accuracy: 0.6216
Epoch 13
Train Loss: 0.8136
Val Accuracy: 0.6216
Epoch 14
Train Loss: 0.7192
Val Accuracy: 0.6216
Epoch 15
Train Loss: 0.8240
Val Accuracy: 0.6486
Epoch 16
Train Loss: 0.7941
Val Accuracy: 0.6486
Early stopping triggered
